In [2]:
!pip install pycountry_convert pandas yfinance tqdm pytz dash plotly jupyter_dash


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\Rinkesh\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [1]:
# fetch_growth_data.py
# (This is your script, enhanced with more robust logging)

import pandas as pd
import yfinance as yf
from tqdm import tqdm
import logging
from datetime import datetime, timedelta
import time
import pytz
import pycountry_convert as pc
import numpy as np

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# Ensure this path is correct for your environment
INPUT_CSV = r"D:\Stock-Market-Indices\data\index_ticker_list_final.csv"
OUTPUT_CSV = "index_growth_data.csv"
YEARS_OF_DATA = 10

# --- (Programmatic Location Derivation is unchanged) ---
TIMEZONE_TO_COUNTRY_CODE = {tz: code for code, tzs in pytz.country_timezones.items() for tz in tzs}

def get_location_from_info(info):
    """Derives continent from ticker info, defaulting to 'Global'."""
    timezone_str = info.get("exchangeTimezoneName")
    if timezone_str in TIMEZONE_TO_COUNTRY_CODE:
        try:
            country_code = TIMEZONE_TO_COUNTRY_CODE[timezone_str]
            continent_code = pc.country_alpha2_to_continent_code(country_code)
            return pc.convert_continent_code_to_continent_name(continent_code)
        except Exception:
            # This can happen for non-standard country codes like 'EU'
            pass
    return "Global"


def fetch_and_process_indices(input_file):
    """
    Fetches historical data for a list of indices, calculates cumulative growth,
    and returns a consolidated DataFrame.
    """
    try:
        df_indices = pd.read_csv(input_file)
    except FileNotFoundError:
        logging.error(f"Input file not found: {input_file}. Please create this file with 'Index Name' and 'Yahoo Finance Ticker' columns.")
        return None

    end_date = datetime.now()
    start_date = end_date - timedelta(days=365.25 * YEARS_OF_DATA)
    all_growth_data = []

    for _, row in tqdm(df_indices.iterrows(), total=df_indices.shape[0], desc="Fetching and Processing Data"):
        ticker_str = row["Yahoo Finance Ticker"]
        original_name = row["Index Name"]

        try:
            ticker_obj = yf.Ticker(ticker_str)
            ticker_info = ticker_obj.info
            continent = get_location_from_info(ticker_info)
            # Use longName or shortName from yfinance if available, otherwise use our name
            best_name = ticker_info.get("longName") or ticker_info.get("shortName") or original_name

            # Fetch the data using the yfinance object for better caching
            hist = ticker_obj.history(start=start_date, end=end_date, interval="1mo", auto_adjust=True)

            # --- DATA VALIDATION ---
            if hist.empty:
                logging.warning(f"No historical data found for {ticker_str} ('{best_name}') in the last {YEARS_OF_DATA} years. Skipping.")
                continue

            # Ensure 'Close' is the price column (auto_adjust=True should handle this)
            price_column = 'Close'
            if price_column not in hist.columns:
                logging.error(f"Failed to process {ticker_str}: '{price_column}' column not found. Available columns: {hist.columns.tolist()}. Skipping.")
                continue

            # Drop any rows where the closing price is missing
            hist.dropna(subset=[price_column], inplace=True)
            if hist.empty:
                logging.warning(f"Data for {ticker_str} is all NaN after dropping NaNs. Skipping.")
                continue

            # --- CRITICAL CALCULATION ---
            # Get the very first valid price point for this specific index's time range.
            initial_price = hist[price_column].iloc[0]

            # If the initial price is zero or non-positive, growth calculation is impossible.
            if not isinstance(initial_price, (int, float)) or initial_price <= 0:
                logging.error(f"Invalid initial price ({initial_price}) for {ticker_str}. Cannot calculate growth. Skipping.")
                continue

            # Calculate cumulative growth based on the first data point
            hist['Cumulative Growth (%)'] = ((hist[price_column] / initial_price) - 1) * 100

            hist['Index Name'] = best_name
            hist['Ticker'] = ticker_str
            hist['Continent'] = continent

            all_growth_data.append(hist.reset_index())
            time.sleep(0.05) # Be polite to the API

        except Exception as e:
            # Catching yfinance download errors or info errors
            logging.error(f"A general error occurred for ticker {ticker_str}: {e}")

    if not all_growth_data:
        logging.error("No valid data could be processed for any index. Output file will not be created.")
        return None

    final_df = pd.concat(all_growth_data, ignore_index=True)

    # Final sanitization for any infinite values that might slip through
    final_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    final_df.dropna(subset=['Cumulative Growth (%)'], inplace=True)

    return final_df

if __name__ == "__main__":
    logging.info(f"Starting data fetching for the last {YEARS_OF_DATA} years...")
    growth_df = fetch_and_process_indices(INPUT_CSV)

    if growth_df is not None and not growth_df.empty:
        # Final sanity check on the generated data
        min_g = growth_df['Cumulative Growth (%)'].min()
        max_g = growth_df['Cumulative Growth (%)'].max()
        logging.info(f"Data processing complete. Final growth range: [{min_g:.2f}%] to [{max_g:.2f}%]")

        growth_df.to_csv(OUTPUT_CSV, index=False)
        logging.info(f"Successfully processed {growth_df['Ticker'].nunique()} unique indices and saved data to {OUTPUT_CSV}")
    else:
        logging.warning("No valid growth data was generated. The output file was not created.")

2025-10-30 17:59:26,666 - INFO - Starting data fetching for the last 10 years...
Fetching and Processing Data: 100%|██████████| 82/82 [00:56<00:00,  1.46it/s]
2025-10-30 18:00:22,832 - INFO - Data processing complete. Final growth range: [-52.59%] to [21499.45%]
2025-10-30 18:00:22,916 - INFO - Successfully processed 77 unique indices and saved data to index_growth_data.csv


In [7]:
import pandas as pd
import dash
from dash import dcc, html, ctx
from dash.dependencies import Input, Output, State

# --- 1. Load and Prepare Data ---
try:
    df = pd.read_csv("index_growth_data.csv")
except FileNotFoundError:
    print("Error: 'index_growth_data.csv' not found. Please run the data fetching script first.")
    exit()

# --- Filter 1: Manually remove specified anomalies ---
indices_to_remove = ['BIST 100', 'MERVAL']
tickers_to_remove = ['XU100.IS']
print(f"Original data rows: {len(df)}")
df = df[~df['Index Name'].isin(indices_to_remove)]
df = df[~df['Ticker'].isin(tickers_to_remove)]
print(f"Data rows after removing specified anomalies: {len(df)}")

# --- Data Prep: Convert to Datetime for calculations ---
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
df.dropna(subset=['Date'], inplace=True)
df.sort_values(by=['Ticker', 'Date'], inplace=True)

# --- Filter 2: Remove indices with stale data (missing at the end) ---
max_date_overall = df['Date'].max()
recency_threshold = max_date_overall - pd.Timedelta(days=60)
df['last_date_for_index'] = df.groupby('Ticker')['Date'].transform('max')
original_indices_count = df['Ticker'].nunique()
df = df[df['last_date_for_index'] >= recency_threshold].copy()
df.drop(columns=['last_date_for_index'], inplace=True)
print(f"\nRemoved {original_indices_count - df['Ticker'].nunique()} indices with stale data.")

# --- Filter 3: Remove indices with internal data gaps ---
df['time_gap'] = df.groupby('Ticker')['Date'].diff()
df['max_gap'] = df.groupby('Ticker')['time_gap'].transform('max')
gap_threshold = pd.Timedelta(days=45)
original_indices_count = df['Ticker'].nunique()
df = df[df['max_gap'] <= gap_threshold].copy()
df.drop(columns=['time_gap', 'max_gap'], inplace=True)
print(f"Removed {original_indices_count - df['Ticker'].nunique()} indices with internal data gaps.")


# =====================================================================================
# === FIX: Standardize Data to Monthly Steps for Smooth Animation =====================
# =====================================================================================
# 1. Create a standardized 'YearMonth' string column (e.g., "2022-01")
df['YearMonth'] = df['Date'].dt.strftime('%Y-%m')

# 2. Ensure only ONE data point per index per month. We keep the latest one within that month.
# This is a robust way to handle data where yfinance might give two points for one month.
df = df.loc[df.groupby(['Ticker', 'YearMonth'])['Date'].idxmax()].copy()

# 3. The main 'Date' column for our app will now be the standardized 'YearMonth'
df['Date'] = df['YearMonth']
df.drop(columns=['YearMonth'], inplace=True)

print(f"\nData successfully standardized to {len(df)} monthly points.")
print(f"Kept {df['Ticker'].nunique()} indices with complete, consistent data for visualization.\n")
# =====================================================================================
# === END OF FIX ======================================================================
# =====================================================================================

# --- Prepare data for Dash components ---
data_for_store = df.to_dict('records')
unique_dates = sorted(df['Date'].unique()) # This list now correctly contains 'YYYY-MM' strings
# Make marks for every year (12 months) for a clean look
date_marks = {i: date[:4] for i, date in enumerate(unique_dates) if i % 12 == 0 or i == len(unique_dates) - 1}

# --- 2. Initialize the App ---
app = dash.Dash(__name__)
server = app.server

# --- 3. App Layout (Unchanged) ---
app.layout = html.Div([
    html.H1("Stock Index Cumulative Growth Over 10 Years"),
    html.Div([
        html.Div(
            dcc.Dropdown(
                id='continent-filter',
                options=[{'label': c, 'value': c} for c in unique_continents] + [{'label': 'All Continents', 'value': 'All'}],
                value=['Europe'], multi=True, placeholder="Select continents"
            ), style={'width': '35%', 'display': 'inline-block', 'verticalAlign': 'top'}
        ),
        html.Div(
            dcc.Dropdown(id='index-highlighter', multi=True, placeholder="Highlight specific indices to focus..."),
            style={'width': '45%', 'display': 'inline-block', 'verticalAlign': 'top', 'paddingLeft': '10px'}
        ),
        html.Div(
            dcc.Dropdown(
                id='yaxis-type', options=[{'label': 'Linear', 'value': 'linear'}, {'label': 'Log', 'value': 'log'}],
                value='linear', clearable=False
            ), style={'width': '18%', 'float': 'right', 'display': 'inline-block', 'verticalAlign': 'top'}
        )
    ]),
    dcc.Graph(id='growth-chart', style={'height': '70vh'}),
    html.Div([
        dcc.Slider(
            id='date-slider', min=0, max=len(unique_dates) - 1, value=len(unique_dates) - 1,
            marks=date_marks, step=1, updatemode='mouseup'
        ),
        html.Button('▶ Play', id='play-button', n_clicks=0, style={'marginRight': '10px'}),
        html.Button('❚❚ Pause', id='pause-button', n_clicks=0)
    ], style={'padding': '20px'}),
    dcc.Store(id='data-store', data=data_for_store),
    dcc.Store(id='unique-dates-store', data=unique_dates),
    dcc.Store(id='filtered-indices-store'),
    dcc.Interval(id='interval-component', interval=200, n_intervals=0, disabled=True)
])

# --- 4. Callbacks (Unchanged) ---
@app.callback(
    Output('filtered-indices-store', 'data'),
    Input('continent-filter', 'value'),
    State('data-store', 'data')
)
def update_available_indices(selected_continents, all_data):
    df_all = pd.DataFrame(all_data)
    if not selected_continents or 'All' in selected_continents: df_filtered = df_all
    else: df_filtered = df_all[df_all['Continent'].isin(selected_continents)]
    unique_indices = sorted(df_filtered['Index Name'].unique())
    return [{'label': idx, 'value': idx} for idx in unique_indices]

app.clientside_callback(
    "function(data) { return data; }",
    Output('index-highlighter', 'options'),
    Input('filtered-indices-store', 'data')
)

@app.callback(
    Output('interval-component', 'disabled'),
    Input('play-button', 'n_clicks'), Input('pause-button', 'n_clicks'),
)
def toggle_animation(play_clicks, pause_clicks):
    if ctx.triggered_id == 'play-button': return False
    return True

@app.callback(
    Output('date-slider', 'value'),
    Input('interval-component', 'n_intervals'),
    State('date-slider', 'value'), State('date-slider', 'max')
)
def advance_slider(n, current_val, max_val):
    if current_val < max_val: return current_val + 1
    return max_val

app.clientside_callback(
    """
    function(slider_value, selected_continents, yaxis_type, highlighted_indices, all_data, unique_dates) {
        // This JS code now receives a clean, monthly unique_dates array, so the animation will be smooth.
        // No changes are needed here.
        const current_date = unique_dates[slider_value];
        let filtered_data = all_data.filter(d => d.Date <= current_date);
        if (selected_continents && selected_continents.length > 0 && !selected_continents.includes('All')) {
            filtered_data = filtered_data.filter(d => selected_continents.includes(d.Continent));
        }
        const lines_data = {};
        const has_highlights = highlighted_indices && highlighted_indices.length > 0;
        filtered_data.forEach(d => {
            const name = d['Index Name'];
            const growth = parseFloat(d['Cumulative Growth (%)']);
            if (!isNaN(growth) && isFinite(growth)) {
                 if (!lines_data[name]) {
                    const is_highlighted = has_highlights && highlighted_indices.includes(name);
                    lines_data[name] = {
                        x: [], y: [], name: `${name} (${d.Continent})`, type: 'scatter', mode: 'lines',
                        opacity: is_highlighted ? 1.0 : (has_highlights ? 0.2 : 0.7),
                        line: { width: is_highlighted ? 3 : 1.5 }
                    };
                }
                lines_data[name].x.push(d.Date);
                lines_data[name].y.push(growth);
            }
        });
        const line_traces = Object.values(lines_data);
        const marker_traces = line_traces.map(trace => ({
            x: [trace.x[trace.x.length - 1]], y: [trace.y[trace.y.length - 1]], name: trace.name,
            type: 'scatter', mode: 'markers', marker: { size: 8, opacity: trace.opacity }, showlegend: false
        }));
        let yAxisRange = null; let yAxisType = yaxis_type;
        const all_growth_values = filtered_data.map(d => parseFloat(d['Cumulative Growth (%)'])).filter(g => !isNaN(g) && isFinite(g));
        if (all_growth_values.length > 0) {
            let data_min = Math.min(...all_growth_values); let data_max = Math.max(...all_growth_values);
            if (yAxisType === 'log') {
                const positive_values = all_growth_values.filter(g => g > 0);
                if (positive_values.length > 0) {
                    let min_positive = Math.min(...positive_values); let max_positive = Math.max(...positive_values);
                    if (min_positive < 0.01 && min_positive > 0) { min_positive = 0.01; } else if (min_positive <= 0) { min_positive = 0.1; }
                    let y_min_log = min_positive * 0.9; let y_max_log = max_positive * 1.1;
                    if (y_max_log <= y_min_log) { y_max_log = y_min_log * 10; if (y_max_log <= y_min_log) y_max_log = y_min_log + 1; }
                    yAxisRange = [y_min_log, y_max_log];
                } else { yAxisType = 'linear'; yAxisRange = null; }
            }
            if (yAxisRange === null) {
                let calculated_min, calculated_max;
                if (data_min === data_max) {
                    calculated_min = data_min - (data_min === 0 ? 10 : Math.abs(data_min) * 0.1);
                    calculated_max = data_max + (data_max === 0 ? 10 : Math.abs(data_max) * 0.1);
                    if (calculated_max - calculated_min < 20) { calculated_min = data_min - 10; calculated_max = data_max + 10; }
                } else {
                    const buffer = (data_max - data_min) * 0.10;
                    calculated_min = data_min - buffer; calculated_max = data_max + buffer;
                }
                if (calculated_min >= calculated_max) { calculated_min = data_min - 1; calculated_max = data_max + 1; }
                yAxisRange = [calculated_min, calculated_max];
            }
        } else { yAxisRange = null; yAxisType = 'linear'; }
        const layout = {
            title: { text: `Cumulative Growth up to ${current_date}`, x: 0.05, xanchor: 'left' },
            xaxis: { title: 'Date', range: [unique_dates[0], unique_dates[unique_dates.length - 1]], autorange: false },
            yaxis: { title: 'Cumulative Growth (%)', type: yAxisType, autorange: (yAxisRange === null), range: yAxisRange },
            hovermode: 'closest',
            legend: { orientation: 'v', x: 1.02, xanchor: 'left', y: 1 },
            margin: { r: 200 }
        };
        return {data: line_traces.concat(marker_traces), layout: layout};
    }
    """,
    Output('growth-chart', 'figure'),
    [Input('date-slider', 'value'), Input('continent-filter', 'value'), Input('yaxis-type', 'value'), Input('index-highlighter', 'value')],
    [State('data-store', 'data'), State('unique-dates-store', 'data')]
)

# --- 5. Run the App ---
if __name__ == '__main__':
    app.run(debug=True)

Original data rows: 8452
Data rows after removing specified anomalies: 8212

Removed 0 indices with stale data.
Removed 9 indices with internal data gaps.

Data successfully standardized to 7817 monthly points.
Kept 66 indices with complete, consistent data for visualization.

